# Business Entity Resolution — training notebook

Cell-by-cell version of `python -m src.run_pipeline`. Runs identically locally (Windows/Mac/Linux, 16 GB
laptop) and on a SageMaker notebook instance (`ml.r5.4xlarge`, 100 GB RAM) — it imports the same
`src/` modules the CLI uses, so results match the CLI exactly and every stage is still parquet-cached
under `cache/`.

**Resynced 2026-09-26** with the current `src/run_pipeline.py` (it had drifted: a stale
`rp.sample_s1` call that no longer exists, a `transfer_report` call missing its now-required
`fit_mask`/`fit_w` arguments, and no hard/easy negative sampling weights at all on `train_oof`/
`fit_full` — all fixed to match the CLI's actual current logic, including the Exp J2 `comp_col`
wiring for the joint decoder). Defaults now point at a full-dataset run with stage 2 on
(`EVAL_ONLY=False`, `FRAC=1.0`, `NO_STAGE2=False`) — this is the run that exercises Exp J1/J2 and
every other Prompt 8A-8H addition for the first time at real scale.

**Layout:** this notebook lives in `code/business_entity_resolution/notebooks/`. It walks up from its
own working directory to find the `src/` package, so it does not matter whether Jupyter's cwd is this
folder or `code/business_entity_resolution/`.

**How to use:**
- Local dev loop: set `EVAL_ONLY = True`, `FRAC = 0.1` (or leave default) — no test-set inference, ~10 min.
- Full run (this session's target / SageMaker): keep the defaults (`EVAL_ONLY=False`, `FRAC=1.0`,
  `NO_STAGE2=False`). Needs ~100 GB RAM at `FRAC=1.0` — on a 16 GB laptop lower `FRAC` even for a full
  run, or run on SageMaker.
- Re-running a cell is safe: every expensive stage is cached (`cached_frame`/`get_translit`/etc.), so a
  kernel restart resumes instead of recomputing.
- On Windows, keep `ER_JOBS` low (3-4). Blocking spawns that many OS processes; more than ~6-8 tends to
  hit `WinError 1450` (insufficient system resources). `ER_LGB_JOBS` (LightGBM threads) is independent
  and safe to set much higher — LightGBM uses threads, not processes. On SageMaker, raise both.


In [ ]:
import os, sys, pathlib

# --- locate the package (works from notebooks/ or business_entity_resolution/, local or SageMaker) ---
_dir = pathlib.Path.cwd().resolve()
while not (_dir / "src").is_dir():
    if _dir == _dir.parent:
        raise RuntimeError("could not find the business_entity_resolution package (looked for a src/ dir)")
    _dir = _dir.parent
PKG_DIR = _dir
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
print("package dir:", PKG_DIR)


In [ ]:
# --- run parameters (edit these, then Run All) ---
DATA_DIR = PKG_DIR / "../../dataset"      # ../../dataset relative to this package, same on SageMaker after `git clone`
OUT_DIR = PKG_DIR / "../../output"
EVAL_ONLY = False        # True: dev OOF only, no test inference. False: also blocks/scores/writes the test set.
FRAC = 1.0               # share of train S1 entities to use; 1.0 = full dataset. None -> config.DEV_FRAC (eval) / config.TRAIN_FRAC (full)
NO_STAGE2 = False         # True: stage-1 only (MVP, historical). False (default): real stage 2 -- also required for Exp J2 (joint decoder) to engage.
NO_TRANSFER = False       # True: skip the US<->India transfer proxy report (saves time; more expensive at FRAC=1.0 -- it refits per country pair).
EXP_ID = "E07"            # non-empty -> appended as a row to experiments/log.csv
CHANGE = "full dataset, stage2 on, Exp J1/J2 (joint S2/S3 clusters + correlated decoder)"

# process/thread split (Windows: keep ER_JOBS <= 4-6; raise ER_LGB_JOBS freely, e.g. to core count).
# On SageMaker (ml.r5.4xlarge, 16 vCPU) both can go higher -- e.g. ER_JOBS=12 ER_LGB_JOBS=16.
os.environ.setdefault("ER_JOBS", "4")
os.environ.setdefault("ER_LGB_JOBS", str(os.cpu_count() or 4))
os.environ.setdefault("ER_CACHE_TAG", "v3")   # v3 = current normalization (alias split + French ets/bis-ter, exp E06)

import platform
print(f"platform={platform.system()} cpu_count={os.cpu_count()} "
      f"ER_JOBS={os.environ['ER_JOBS']} ER_LGB_JOBS={os.environ['ER_LGB_JOBS']} ER_CACHE_TAG={os.environ['ER_CACHE_TAG']}")
try:
    import psutil
    print(f"RAM total={psutil.virtual_memory().total / 1e9:.1f} GB "
          f"available={psutil.virtual_memory().available / 1e9:.1f} GB")
except ImportError:
    pass

In [ ]:
# env vars above must be set before this import (config.py reads them at import time)
import time
import numpy as np
import pandas as pd

from src import blocking, config, context, decide, evaluate, features, io_utils, model, noise, normalize, sampling, translit
from src import run_pipeline as rp

print("cache dir:", config.CACHE_DIR, "| cache tag:", config.CACHE_TAG)
print("N_JOBS (worker processes):", config.N_JOBS, "| LGB_JOBS (LightGBM threads):", config.LGB_JOBS)

## 1. Normalize + learned transliteration (train)
Cached after the first run.

In [ ]:
t0 = time.time()
table = rp.get_translit(DATA_DIR)
s1n, pooln = rp.load_split_norm(DATA_DIR, "train", table)
print(f"train S1={len(s1n):,} pool(S2+S3)={len(pooln):,}")
s1n.head()


## 2. Sample S1 entities + positive pairs

In [ ]:
gt = io_utils.load_ground_truth(DATA_DIR)
pos_all = rp.gt_rid_pairs(gt, s1n, pooln)

frac = FRAC if FRAC is not None else (config.DEV_FRAC if EVAL_ONLY else config.TRAIN_FRAC)
n_matches = np.bincount(pos_all["s1_rid"].to_numpy(), minlength=len(s1n))
rids = sampling.stratified_s1(s1n, n_matches, frac)
pos = pos_all[np.isin(pos_all["s1_rid"].to_numpy(), rids)]
rp.log(f"train entities={len(rids):,} positives={len(pos):,}")


## 3. Blocking (train)
Candidate generation — the ceiling on achievable recall.

In [ ]:
cands = rp.cached_frame(f"cands_train_{frac}", lambda: blocking.generate_candidates(s1n, pooln, rids))
brep = rp.blocking_summary(cands, pos, len(rids))
rp.log(f"blocking {brep}")
row = {"exp_id": EXP_ID, "change": CHANGE,
       "block_recall": round(brep["block_recall"], 4), "cands_per_entity": round(brep["cands_per_entity"], 1)}
row


## 4. Stage-1 features

In [ ]:
generic = rp.get_generic_vocab(s1n, pooln, pos_all)
feats = rp.cached_frame(f"feats1_train_{frac}", lambda: features.build_features(cands, s1n, pooln, generic))
y = np.isin(rp.pair_key(feats["s1_rid"], feats["pool_rid"]), rp.pair_key(pos["s1_rid"], pos["pool_rid"])).astype(np.int8)
groups = feats["s1_rid"].to_numpy()
cols1 = model.feature_cols(feats)
fit_mask, fit_w = sampling.training_rows(feats["f_block_rank"].to_numpy(), y)
fit = dict(train_mask=fit_mask, sample_weight=fit_w)
rp.log(f"stage1: {len(feats):,} pairs ({fit_mask.sum():,} fitted), {len(cols1)} features, pos rate {y.mean():.3f}")


## 5. Stage-1 LightGBM (5-fold GroupKFold OOF)
~20-25 min on a laptop at `FRAC=0.1`.

In [ ]:
p1, models1 = model.train_oof(feats[cols1], y, groups, **fit)
feats["p1"] = p1


In [ ]:
# feature importance (first fold model) - purely diagnostic, notebook-only
try:
    import matplotlib.pyplot as plt
    imp = pd.Series(models1[0].feature_importances_, index=cols1).sort_values(ascending=False).head(15)
    imp[::-1].plot.barh(figsize=(6, 5), title="stage-1 top 15 feature importances (fold 0)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping plot")


## 6. Stage 2 (optional): group context, competition, triangle support
Skipped when `NO_STAGE2 = True` (the MVP path).

In [ ]:
cols, pcol, models2 = cols1, "p1", None
if not NO_STAGE2:
    feats = context.add_stage2_features(feats, pooln)
    cols = cols1 + model.feature_cols(feats, prefix="f2_")
    rp.log(f"stage2: {len(cols)} features")
    p2, models2 = model.train_oof(feats[cols], y, groups, **fit)
    feats["p2"] = p2
    pcol = "p2"
else:
    rp.log("stage2 skipped (NO_STAGE2=True)")


## 7. Decode: calibrate -> tune threshold / expected-F0.5 -> OOF breakdown

In [ ]:
scorer = rp.Scorer(s1n, pooln, gt, rids)
extra = brep["missed_per_entity"]
scored = feats[["s1_rid", "pool_rid"]].assign(p=feats[pcol].to_numpy())
# Exp J2 (docs/RESEARCH_JOINT_S2S3.md): only available once stage 2 has run.
comp_col = context.CLUSTER_ID_COL if pcol == "p2" else None
if comp_col:
    scored[comp_col] = feats[comp_col].to_numpy()
calib = decide.fit_calibrator(scored["p"].to_numpy(), y)
dec, grid = decide.tune(scored, calib, extra, scorer, comp_col=comp_col)
rp.log(f"decoder {dec}")
sel = decide.apply_decoder(scored, dec, calib, extra, comp_col=comp_col)
bd = scorer.breakdown(sel)
rp.log(f"OOF breakdown {bd}")
bd


## 8. Transfer report (France proxy: train US -> score India, and back)
Skipped when `NO_TRANSFER = True`.

In [ ]:
tr = {} if NO_TRANSFER else rp.transfer_report(feats, y, cols1, scorer, dec, extra, fit_mask, fit_w)
rp.log(f"transfer {tr}")
tr


## 9. Log this experiment
Appends a row to `experiments/log.csv` only if `EXP_ID` is set.

In [ ]:
row.update({"f05_all": round(bd["all"], 5), "f05_us": round(bd.get("country=US", np.nan), 5),
            "f05_in": round(bd.get("country=India", np.nan), 5),
            "f05_singletons": round(bd["singletons"], 5), "f05_matched": round(bd["matched"], 5),
            "us_to_in": round(tr.get("US->India", np.nan), 5), "in_to_us": round(tr.get("India->US", np.nan), 5),
            "t_match": dec.get("t_match", ""), "t_empty": dec.get("t_empty", "")})
if EXP_ID:
    rp.append_log(row)
    rp.cache_path("decoder", "json").write_text(__import__("json").dumps(dec))
    print("logged:", row)
else:
    print("EXP_ID empty; not writing to experiments/log.csv. Row would be:")
    print(row)


## 10. Test inference (skipped when `EVAL_ONLY = True`)

Refits on all sampled train rows, then blocks + scores the **test** set and writes
`output/matching_results.tsv` + `output/candidate_pairs.tsv`. This is the RAM-heavy part
(~100 GB recommended for the full `TRAIN_FRAC`; keep `FRAC` small on a laptop) and the part that
previously crashed on Windows with `WinError 1450` — that's what `ER_JOBS` (kept low above) fixes.


In [ ]:
if EVAL_ONLY:
    print("EVAL_ONLY=True; stopping here (no test inference). Set EVAL_ONLY=False and re-run to produce output/.")
else:
    rp.log("fitting full models for test")
    m1 = model.fit_full(feats[cols1], y, n_estimators=model.mean_best_iter(models1), **fit)
    m2 = model.fit_full(feats[cols], y, n_estimators=model.mean_best_iter(models2), **fit) if models2 else None


In [ ]:
if not EVAL_ONLY:
    import gc
    # free train-only memory before test inference, same as the CLI path
    del feats, scored, sel
    gc.collect()
    s1t, poolt = rp.load_split_norm(DATA_DIR, "test", table)
    candt = rp.cached_frame("cands_test", lambda: blocking.generate_candidates(s1t, poolt))
    rp.log(f"test candidates {len(candt):,} ({len(candt) / len(s1t):.1f}/entity)")


In [ ]:
if not EVAL_ONLY:
    if m2 is None:
        scored_t = rp.score_stream(candt, s1t, poolt, generic, m1, cols1)
    else:
        ft = rp.cached_frame("feats1_test", lambda: features.build_features(candt, s1t, poolt, generic))
        ft["p1"] = m1.predict_proba(ft[cols1])[:, 1]
        ft = context.add_stage2_features(ft, poolt)
        scored_t = ft[["s1_rid", "pool_rid"]].assign(p=m2.predict_proba(ft[cols])[:, 1])
        if comp_col:
            scored_t[comp_col] = ft[comp_col].to_numpy()


## 11. Write outputs + validate

In [ ]:
if not EVAL_ONLY:
    selt = decide.apply_decoder(scored_t, dec, calib, extra, comp_col=comp_col if m2 is not None else None)
    rp.write_outputs(OUT_DIR, s1t, poolt, candt[["s1_rid", "pool_rid"]], selt)
    code_ = rp.validate(OUT_DIR, DATA_DIR)
    rp.log(f"validator exit={code_}; matched entities={selt['s1_rid'].nunique():,}/{len(s1t):,}; "
           f"done in {time.time() - t0:.0f}s")


## 12. Save models + decoder locally


In [ ]:
import json

MODEL_DIR = config.CACHE_DIR
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not EVAL_ONLY:
    m1.booster_.save_model(str(MODEL_DIR / "m1_stage1.txt"))
    if m2 is not None:
        m2.booster_.save_model(str(MODEL_DIR / "m2_stage2.txt"))
    else:
        rp.log("m2 is None (models2 empty) -- stage-2 context model not trained this run, skipping save")

    decoder_path = MODEL_DIR / f"{config.CACHE_TAG}_decoder.json"
    decoder_path.write_text(json.dumps(dec))
    rp.log(f"saved models + decoder -> {MODEL_DIR}")
